## Implementing the BPE algorithm
The aim of this exercise is to implement the BPE tokenization algorithm. As a reminder, the principle consists in gathering the words or “tokens” that appear the most times in succession.

For example, if we consider the corpus containing the words in the following table (with the number of occurrences of each word):

| words | occurrence |
|------|-----------|
| voting | 2 |
| vote | 3 |
| slow | 1 |
| slowly | 2 |


And if the initial “tokens” are the letters of the alphabet, then the prefix “vo” will initially be added to the list of sub-words (tokens), cause the bigram "v" "o" occurs 5 times (2 + 3).

The steps involved in implementing the BPE algorithm are as follows:
1. Download a text corpus (here a wikipedia page)
2. Cut the text into words (using the “space” and “ponctuation” characters) and count the number of occurrences of each word.
3. Initialize the word dictionary with the initial tokens (letters of the alphabet)
4. Run BPE algorithm (learn vocabulary)
5. Test token decomposition on selected sentences (apply learned rules)


In the documents you will observe a lot of TODO in the code, replace it by your code.


### The class we will complete


At the end we will create an object (python) class as it follows :

```python
class Tokenizer:
    ''' 
        A class for our Tokenizer

        Methods
        -------
        fit(text_corpus: str) : None
            Fit the tokenizer based on the input text corpus 
        tokenize(text: str): List[str]
            Tokenize a text and return the list of token ids
        detokenize(tokens: List[str]): str
            From a list of token ids return the corresponding text
    '''

    def __init__(self, vocabulary_size: int = 500):
        ''' 
            Parameters
            ----------
            vocabulary_size : int
                The expected size of the vocabulary

        '''
        super().__init__()
        self.vs = vocabulary_size

    def fit(self, text_corpus: str):
        ''' Train the tokenizer on the provided text.

            Parameters
            ----------
            text_corpus : str
                The text used to train the model
        '''
        raise NotImplementedError


    def tokenize(self, text: str) -> List[str]:
        ''' Tokenize a text.

            Parameters
            ----------
            text : str
                The text to tokenize

            Returns
            -------
            List[str]
                The list of tokens
        '''
        raise NotImplementedError

    def detokenize(self, tokens : List[int]) -> str:
        ''' Reverse the tokenization.

            Parameters
            ----------
            tokens : List[str]
                A list of token ids

            Returns
            -------
            str
                The text corresponding to tokens
        '''
        raise NotImplementedError

```

## Requirement

**For this lab you only need the python standard library :D**

In [1]:
import re # the regex library
import json # read export json format

from typing import List # to specify the types in function def
from collections import Counter # a tools to counts unique occurences

from urllib.request import urlopen, Request

### Step 1: Download a corpus

For this lab we will consider a wikipedia page in french [Grèce antique](https://fr.wikipedia.org/w/api.php?format=json&action=query&prop=extracts&explaintext&redirects=1&titles=Gr%C3%A8ce_antique), but you are free to choose any content you want !

In [2]:


url_request  = 'https://fr.wikipedia.org/w/api.php?format=json&action=query&prop=extracts&explaintext&redirects=1&titles=Gr%C3%A8ce_antique'
wikipedia_request = Request(url_request)
wikipedia_request.add_header("User-Agent", "Course (thomas.gerald@lisn.fr)")
raw_page = urlopen(wikipedia_request)
json_page = json.load(raw_page)


In [3]:
corpus = list(json_page['query']['pages'].values())[0]['extract']


### Step 2: Splitting text into words (or sequence of character no containing space)

To split the text into words, we'll use the following regex ```r'(\b[^\s]+\b)'```. To count words, we'll use python's Counter object. 
1. Store each word and its number of occurrences in **count_words**.
2. Give the 10 most frequent words (you'll store them in most_commons_words).

In [4]:

word_regex = re.compile(r'(\b[^\s]+\b)')
#words = word_regex.findall(corpus)
words = ['▁' + w for w in word_regex.findall(corpus)] # Added _ to allow detokenization of the text after BPE tokenization

count_words = dict()
most_commons_words = list()

for word in words:
    if word in count_words:
        count_words[word] += 1
    else:
        count_words[word] = 1

most_commons_words = Counter(count_words).most_common(10)
print(most_commons_words)

[('▁de', 1691), ('▁la', 1100), ('▁et', 977), ('▁des', 927), ('▁les', 773), ('▁à', 601), ('▁le', 520), ('▁en', 480), ('▁qui', 378), ('▁du', 352)]


### Step 3: Initialize word dictionary with initial tokens (letters of the alphabet)

Create the initial vocabulary in the vocab variable. How many initial tokens do you have?

In [5]:
alphabet = " ".join(sorted(set("".join(words))))
vocab = alphabet.split()
print(vocab)

["'", '(', ')', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '²', 'À', 'Â', 'É', 'à', 'â', 'ç', 'è', 'é', 'ê', 'î', 'ï', 'ó', 'ô', 'ù', 'û', 'ü', 'Œ', 'œ', 'Α', 'Γ', 'Ε', 'Ι', 'Ο', 'Υ', 'Φ', 'Χ', 'Ψ', 'Ω', 'α', 'ι', 'κ', 'ρ', 'ς', 'ό', '’', '▁']


### Step 4: Learning the tokenizer
To learn the tokenizer we need several functions:
1. A function to calculate the frequency of each token pair.
2. A function to merge a pair

Several variables will be required:
1. **vocab** containing current vocabulary
2. **merge_rules** containing all the merge rules (a dictionary containing as key a pair of tokens to merge and the result of the token merge). For example: {('e', 's'), 'es', ('en', 't') :'ent'}.
3. **splits** A dictionary containing the current breakdown of the corpus, with the word as key and the list of “tokens” as value.


In [6]:
# In the first step splits contains the words broken down into characters
splits = dict()
for word in words:
    splits[word] = list(word)

print(splits)

{'▁La': ['▁', 'L', 'a'], '▁Grèce': ['▁', 'G', 'r', 'è', 'c', 'e'], '▁antique': ['▁', 'a', 'n', 't', 'i', 'q', 'u', 'e'], '▁est': ['▁', 'e', 's', 't'], '▁une': ['▁', 'u', 'n', 'e'], '▁civilisation': ['▁', 'c', 'i', 'v', 'i', 'l', 'i', 's', 'a', 't', 'i', 'o', 'n'], '▁de': ['▁', 'd', 'e'], "▁l'Antiquité": ['▁', 'l', "'", 'A', 'n', 't', 'i', 'q', 'u', 'i', 't', 'é'], '▁des': ['▁', 'd', 'e', 's'], '▁peuples': ['▁', 'p', 'e', 'u', 'p', 'l', 'e', 's'], '▁langue': ['▁', 'l', 'a', 'n', 'g', 'u', 'e'], '▁et': ['▁', 'e', 't'], '▁culture': ['▁', 'c', 'u', 'l', 't', 'u', 'r', 'e'], '▁grecques': ['▁', 'g', 'r', 'e', 'c', 'q', 'u', 'e', 's'], '▁développée': ['▁', 'd', 'é', 'v', 'e', 'l', 'o', 'p', 'p', 'é', 'e'], '▁en': ['▁', 'e', 'n'], '▁dans': ['▁', 'd', 'a', 'n', 's'], '▁la': ['▁', 'l', 'a'], '▁partie': ['▁', 'p', 'a', 'r', 't', 'i', 'e'], '▁occidentale': ['▁', 'o', 'c', 'c', 'i', 'd', 'e', 'n', 't', 'a', 'l', 'e'], "▁l'Asie": ['▁', 'l', "'", 'A', 's', 'i', 'e'], '▁Mineure': ['▁', 'M', 'i', 'n', 

#### Compute the frequency of token pairs
Create a function **compute_pair_freqs**, which, given the words broken down into tokens (splits dictionary) and the frequency of the words, returns the frequency of each pair of tokens (note only successive sub-words).

In [7]:
def compute_pair_freqs(splits, word_freqs):
    pair_freqs = {}
    for word, freq in word_freqs.items():
        #print(f"word: {word}, freq: {freq}")
        for i in range(len(splits[word]) - 1):
            pair = (splits[word][i], splits[word][i + 1])
            if pair in pair_freqs:
                pair_freqs[pair] += freq
            else:
                pair_freqs[pair] = freq
    
    return pair_freqs

In [8]:
pair_freqs = compute_pair_freqs(splits, count_words)
{k: pair_freqs[k] for k  in list(pair_freqs.keys())[:5]}

{('▁', 'L'): 576,
 ('L', 'a'): 165,
 ('▁', 'G'): 270,
 ('G', 'r'): 256,
 ('r', 'è'): 244}

#### Find the most frequent pair and merge a pair
1. Create a function **most_frequent(pair_freqs)** returning the most frequent pair of tokens.
2. Create a **merge_pair()** function which, given a pair, returns the new splits of the corpus.

In [9]:
def most_frequent(pair_freqs):
    if not pair_freqs:
        return None
    return max(pair_freqs, key=pair_freqs.get)

freq_pair = most_frequent(pair_freqs)
print(freq_pair)
print(pair_freqs[freq_pair])

('e', 's')
5657


In [10]:
def merge_pair(a : str, b: str, splits):
    '''
        splits : the dataset of words tokenized
        a : the first token
        b : the second token

        return : a new splits where every (a, b) pair has been merged
    '''
    new_splits = {}
    for word, previous_split in splits.items():
        new_split = []
        i = 0
        while i < len(previous_split):
            if (i < len(previous_split) - 1
                    and previous_split[i] == a
                    and previous_split[i + 1] == b):
                new_split.append(a + b)
                i += 2          # skip both tokens we just merged
            else:
                new_split.append(previous_split[i])
                i += 1
        new_splits[word] = new_split
    return new_splits

In [11]:
freq_pair = most_frequent(pair_freqs)
print(freq_pair)
new_splits = merge_pair(freq_pair[0], freq_pair[1], splits)

('e', 's')


#### Apply the algorithm until the desired vocabulary size is reached.
Create a BPE object that takes as arguments a corpus, a vocabulary size and train the BPE algorithm. The algorithm stores the final vocabulary in the **vocabulary** attribute and the merge rules in **merge_rules**.
For merge_rules, here's an example of its contents:
```
{('e', 's'): 'es',
 ('n', 't'): 'nt',
 ('q', 'u'): 'qu',
 ('r', 'e'): 're',
 ('o', 'n'): 'on',
 ('d', 'e'): 'de',
 ('l', 'e'): 'le',
 ('t', 'i'): 'ti',
 ('l', 'a'): 'la',
 ('i', 's'): 'is',
 ('e', 'nt'): 'ent', ...
 }
```

In [12]:
class BPE:
    def __init__(self, corpus, vocabulary_size=500):
        super().__init__()
        self.word_regex = re.compile(r'(\b[^\s]+\b)')
        #words = self.word_regex.findall(corpus)
        words = ['▁' + w for w in self.word_regex.findall(corpus)]

        # counting words
        count_words = Counter(words)
        # create initial vocab
        self.vocab = list({char for word in count_words.keys() for char in word})
        self.vocab.sort()
        # create the initial split
        splits = {word: [c for c in word] for word in count_words.keys()}
        # initialise merge_rules
        self.merge_rules = {}

        while len(self.vocab) < vocabulary_size:
            pair_freqs = compute_pair_freqs(splits, count_words)
            most_frequent_pair = most_frequent(pair_freqs)
            if most_frequent_pair is None:
                # nothing left to merge: every word is a single token
                break
            a, b = most_frequent_pair
            self.merge_rules[(a, b)] = a + b
            self.vocab.append(a + b)          # the merged token joins the vocabulary
            splits = merge_pair(a, b, splits)

        self.splits = splits

    def tokenize(self, text):
        #words = self.word_regex.findall(text)
        words = ['▁' + w for w in self.word_regex.findall(text)]
        splits = [[l for l in word] for word in words]
        for pair, merge in self.merge_rules.items():
            for idx, split in enumerate(splits):
                i = 0
                while i < len(split) - 1:
                    if split[i] == pair[0] and split[i + 1] == pair[1]:
                        split = split[:i] + [merge] + split[i + 2:]
                    else:
                        i += 1
                splits[idx] = split
        return sum(splits, [])

In [13]:
my_bpe = BPE(corpus, vocabulary_size=500)
print(len(my_bpe.vocab), "tokens,", len(my_bpe.merge_rules), "merge rules")

500 tokens, 394 merge rules


In [14]:
texte = '''cultures grecques développée en Grèce '''
my_bpe.tokenize(texte)[:12]

['▁cul', 't', 'ur', 'es', '▁grecques', '▁développ', 'ée', '▁en', '▁Grèce']

#### Test by modifying parameters or corpus
Test the algorithm with different hyper-parameters or data

## Using sentencepiece
We're now going to use the `sentencepiece` library, which can be installed (if not already installed) with pip : 

`! pip install sentencepiece`

To train the tokenizer, we'll use the `train` function of `SentencePieceTrainer`. 

In [15]:
import sentencepiece as spm

with open("test-cours.txt", "w") as f:
    f.write(corpus)
spm.SentencePieceTrainer.train(input="test-cours.txt", model_type='BPE',  model_prefix='m', vocab_size=500)

sentencepiece_trainer.cc(77) LOG(INFO) Starts training with : 
trainer_spec {
  input: test-cours.txt
  input_format: 
  model_prefix: m
  model_type: BPE
  vocab_size: 500
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noise_level: 0
  differ

Looking at the “m.vocab” file, what are the differences with the vocabulary learned with your implementation? What modifications could you consider to obtain a similar vocabulary?

In [16]:
# Load the sentencepiece vocabulary (one "token<TAB>score" per line, in merge order)
with open("m.vocab", encoding="utf-8") as f:
    sp_vocab = [line.split("\t")[0] for line in f]

sp_special = [t for t in sp_vocab if t in ("<unk>", "<s>", "</s>")]
sp_chars   = [t for t in sp_vocab if len(t) == 1]
sp_merges  = [t for t in sp_vocab if len(t) > 1 and t not in sp_special]

our_chars  = [t for t in my_bpe.vocab if len(t) == 1]
our_merges = [t for t in my_bpe.vocab if len(t) > 1]

# 1. Composition of the two vocabularies
print("                 ours   sentencepiece")
print(f"special tokens   {0:>4}   {len(sp_special):>4}   {sp_special}")
print(f"single chars     {len(our_chars):>4}   {len(sp_chars):>4}")
print(f"merged tokens    {len(our_merges):>4}   {len(sp_merges):>4}")
print(f"shared tokens    {len(set(my_bpe.vocab) & set(sp_vocab))} / 500")

# 2. Rare characters: sentencepiece keeps only the chars covering 99.95% of the text
print("\nChars only in ours:", sorted(set(our_chars) - set(sp_chars)))
print("Chars only in sp:  ", sorted(set(sp_chars) - set(our_chars)))

# 3. Merge order: both learn the same most frequent pairs first
print("\nFirst 20 merges")
for ours, sp in zip(our_merges[:20], sp_merges[:20]):
    print(f"  {ours:<10} {sp:<10} {'' if ours == sp else '<- differs'}")

# 4. Punctuation: our regex drops it, sentencepiece learns tokens with it
is_punct = lambda t: re.search(r"[^\w▁]", t) is not None
print("\nMerged tokens with punctuation, ours:", [t for t in our_merges if is_punct(t)])
print("Merged tokens with punctuation, sp:  ", [t for t in sp_merges if is_punct(t)])

dropped = Counter(c for c in word_regex.sub("", corpus) if not c.isspace())
print("\nCharacters thrown away by our regex:", dropped.most_common(10))

# 5. Same sentence through both tokenizers
sp_model = spm.SentencePieceProcessor(model_file="m.model")
sentence = "Les cités grecques développent la démocratie (Ve siècle av. J.-C.), à l'époque classique."
print("\nours:", my_bpe.tokenize(sentence))
print("sp:  ", sp_model.encode(sentence, out_type=str))


                 ours   sentencepiece
special tokens      0      3   ['<unk>', '<s>', '</s>']
single chars      106     83
merged tokens     394    414
shared tokens    456 / 500

Chars only in ours: ['K', 'Q', 'W', 'Y', 'Z', 'w', '²', 'Â', 'ó', 'û', 'ü', 'Œ', 'Α', 'Γ', 'Ε', 'Ι', 'Ο', 'Υ', 'Φ', 'Χ', 'Ψ', 'Ω', 'α', 'ι', 'κ', 'ρ', 'ς', 'ό']
Chars only in sp:   [':', ';', '=', '«', '»']

First 20 merges
  es         es         
  ▁d         ▁d         
  ▁l         ▁l         
  nt         nt         
  ▁p         ▁p         
  qu         qu         
  re         re         
  ▁c         ▁c         
  on         on         
  ▁s         ▁s         
  ▁e         ▁e         
  ▁a         ▁a         
  ▁de        ▁de        
  ti         ti         
  is         is         
  ent        ent        
  ur         ur         
  que        que        
  it         it         
  an         an         

Merged tokens with punctuation, ours: ["▁l'", "▁d'", "▁l'é", '.-', '▁J.-', '▁J.-C', "▁l'époque"

The two vocabularies are actually very close: they share 456 tokens out of 500, and the first 20 merges are exactly the same (`es`, `▁d`, `▁l`, `nt`, `qu`, ...). So both learn the same frequent pairs, in the same order. The differences do not come from the algorithm, but from what we do with the text before and around it:

1. **Special tokens**: m.vocab starts with `<unk>`, `<s>` and `</s>` (unknown token, start and end of sentence). We have none.
2. **Rare characters**: sentencepiece only keeps the characters covering 99.95% of the text (83 characters) and replaces the others by `<unk>`. We keep all 106, even the ones appearing once (`α`, `ρ`, `K`, `ü`, ...). Those 23 extra characters take vocabulary slots, so we only learn 394 merges instead of 414.
3. **Punctuation**: our regex removes it before training (2666 commas, 1510 dots, 414 parentheses...), so we can neither learn it nor restore it when detokenizing. sentencepiece keeps it and learns tokens like `▁(`, `),`, `▁«`, `==`.
4. **Merges inside words**: sentencepiece never merges a letter with a punctuation character, while our words keep their apostrophes and dots. That is why we learn `▁l'époque` and `▁J.-C`, where sentencepiece learns `époque` alone and can reuse it after `l'`, `d'`, etc.

On the test sentence, we lose the parentheses and the final dot:
```
ours: ... '▁av', '▁J.-C', '▁à', "▁l'époque", '▁classique'
sp:   ... '▁av', '.', '▁J', '.-', 'C', '.', '),', '▁à', '▁l', "'", 'époque', '▁classique', '.'
```

**To obtain a similar vocabulary we could:**
- keep the punctuation but separate it from the words, and put the `▁` marker only on the chunks that come after a space;
- count the characters and keep only the most frequent ones (99.95% of the occurrences), replacing the others by `<unk>`;
- add `<unk>`, `<s>` and `</s>` at the beginning of the vocabulary and count them in the vocabulary size.

Note: we already added the `▁` marker before each word (see the Detokenization part). Without it, none of our tokens would show the start of a word, and that was the biggest difference with m.vocab.


## Detokenization

Propose and implement a methods to detokenize encoded sentence (you should want to extend your vocabulary)

In [17]:
# Went back and added the underscore to the words in the corpus to allow detokenization 
def detokenize(tokens):
    return "".join(tokens).replace("▁", " ").strip()

print(detokenize(my_bpe.tokenize(texte)))

cultures grecques développée en Grèce
